# 🧪 Sesión 02 · Gestión en GitHub Web y Métodos HTTP (GET, POST, PUT, DELETE)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué no basta con usar POST para todo?

---

## 🖥️ Así se verá el resultado final

```http
$ curl -X POST http://127.0.0.1:8000/api/libros \
       -H "Content-Type: application/json" \
       -d '{"titulo":"Cien años de soledad","anio":1967}'
HTTP/1.1 201 Created
{"id":1,"titulo":"Cien años de soledad","anio":1967}

$ curl http://127.0.0.1:8000/api/libros
HTTP/1.1 200 OK
{"libros":[{"id":1,"titulo":"Cien años de soledad","anio":1967}]}

$ curl -X PUT http://127.0.0.1:8000/api/libros/1 \
       -H "Content-Type: application/json" \
       -d '{"titulo":"Cien años de soledad (edición conmemorativa)","anio":1967}'
HTTP/1.1 200 OK
{"id":1,"titulo":"Cien años de soledad (edición conmemorativa)","anio":1967}

$ curl -X DELETE http://127.0.0.1:8000/api/libros/1
HTTP/1.1 204 No Content
```

_La colección se guarda en memoria: no necesitas base de datos todavía._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Distinguir el significado de GET, POST, PUT, PATCH y DELETE.
- Implementar un CRUD mínimo sobre una colección de libros.
- Verificar la diferencia entre 404 (no existe) y 405 (método no permitido).

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] La colección inicia vacía → HTTP 200
- [ ] POST crea el libro con id 1 → HTTP 201
- [ ] GET lista el libro creado → HTTP 200
- [ ] GET por id encuentra el libro → HTTP 200
- [ ] PUT reemplaza el libro → HTTP 200
- [ ] DELETE elimina el libro → HTTP 204
- [ ] El libro eliminado ya no existe → HTTP 404
- [ ] PATCH no está implementado: 405, no 404 → HTTP 405

## 🧩 El reto

Implementa un CRUD completo de libros en memoria. La colección empieza vacía y cada operación debe devolver el código HTTP que le corresponde.

**Requisitos del contrato**

- GET /api/libros responde 200 con {"libros": [...]}.
- POST /api/libros responde 201 con el libro creado y un id asignado por el servidor.
- GET /api/libros/{id} responde 200 con el libro o 404 si no existe.
- PUT /api/libros/{id} reemplaza el libro y responde 200; DELETE responde 204 sin cuerpo.
- Un verbo no implementado sobre una ruta existente debe responder 405, no 404.

> **Pista conceptual:** En FastAPI, si defines GET/PUT/DELETE para `/api/libros/{id}` pero no PATCH, el framework responde 405 automáticamente. No programes ese caso: deja que el contrato hable.

## 📦 Entregable

Cuaderno ejecutado de principio a fin y bitácora `auditorias/auditoria_sesion02.md`.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [6]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.16
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [7]:

%%writefile app.py
from fastapi import FastAPI, HTTPException, Response, status
from pydantic import BaseModel

app = FastAPI(title="API de libros")

# Modelo para recibir y validar los datos de un libro
class Libro(BaseModel):
    titulo: str
    anio: int


# Colección en memoria: empieza vacía
libros = {}
siguiente_id = 1


# GET: consultar todos los libros
@app.get("/api/libros")
def listar_libros():
    return {"libros": list(libros.values())}


# POST: crear un libro
@app.post("/api/libros", status_code=status.HTTP_201_CREATED)
def crear_libro(libro: Libro):
    global siguiente_id

    nuevo_libro = {
        "id": siguiente_id,
        "titulo": libro.titulo,
        "anio": libro.anio
    }

    libros[siguiente_id] = nuevo_libro
    siguiente_id += 1

    return nuevo_libro


# GET: consultar un libro por su ID
@app.get("/api/libros/{id}")
def obtener_libro(id: int):
    if id not in libros:
        raise HTTPException(status_code=404, detail="Libro no encontrado")

    return libros[id]


# PUT: reemplazar los datos de un libro existente
@app.put("/api/libros/{id}")
def actualizar_libro(id: int, libro: Libro):
    if id not in libros:
        raise HTTPException(status_code=404, detail="Libro no encontrado")

    libros[id] = {
        "id": id,
        "titulo": libro.titulo,
        "anio": libro.anio
    }

    return libros[id]


# DELETE: eliminar un libro
@app.delete("/api/libros/{id}", status_code=status.HTTP_204_NO_CONTENT)
def eliminar_libro(id: int):
    if id not in libros:
        raise HTTPException(status_code=404, detail="Libro no encontrado")

    del libros[id]
    return Response(status_code=status.HTTP_204_NO_CONTENT)


Overwriting app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Estoy trabajando en Google Colab con Python y FastAPI. La información se almacena en memoria y no se utiliza una base de datos. |
| **Rol** | Actuar como desarrollador web con experiencia en APIs REST, FastAPI y métodos HTTP. |
| **Especificación** | Implementar un CRUD de libros con los campos id, titulo y anio. Crear las rutas GET, POST, PUT y DELETE según el contrato, validar los datos y respetar los códigos HTTP 200, 201, 204, 404 y 405. |
| **Output** | Entregar el código completo de app.py, compatible con FastAPI, y explicar cómo comprobar las rutas y documentar los resultados de la verificación.|

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [8]:
from verificar import probar_http

casos = [{'descripcion': 'La colección inicia vacía',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': []}},
 {'descripcion': 'POST crea el libro con id 1',
  'metodo': 'POST',
  'ruta': '/api/libros',
  'json': {'titulo': 'Cien años de soledad', 'anio': 1967},
  'estado': 201,
  'json_contiene': {'id': 1, 'titulo': 'Cien años de soledad', 'anio': 1967}},
 {'descripcion': 'GET lista el libro creado',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': [{'id': 1, 'titulo': 'Cien años de soledad', 'anio': 1967}]}},
 {'descripcion': 'GET por id encuentra el libro',
  'metodo': 'GET',
  'ruta': '/api/libros/1',
  'estado': 200,
  'json_contiene': {'id': 1, 'titulo': 'Cien años de soledad'}},
 {'descripcion': 'PUT reemplaza el libro',
  'metodo': 'PUT',
  'ruta': '/api/libros/1',
  'json': {'titulo': 'Cien años de soledad (edición conmemorativa)', 'anio': 1967},
  'estado': 200,
  'json_contiene': {'id': 1, 'titulo': 'Cien años de soledad (edición conmemorativa)'}},
 {'descripcion': 'DELETE elimina el libro', 'metodo': 'DELETE', 'ruta': '/api/libros/1', 'estado': 204},
 {'descripcion': 'El libro eliminado ya no existe', 'metodo': 'GET', 'ruta': '/api/libros/1', 'estado': 404},
 {'descripcion': 'PATCH no está implementado: 405, no 404',
  'metodo': 'PATCH',
  'ruta': '/api/libros/1',
  'estado': 405}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 8 casos · app.py
✅ 01. La colección inicia vacía
✅ 02. POST crea el libro con id 1
✅ 03. GET lista el libro creado
✅ 04. GET por id encuentra el libro
✅ 05. PUT reemplaza el libro
✅ 06. DELETE elimina el libro
✅ 07. El libro eliminado ya no existe
✅ 08. PATCH no está implementado: 405, no 404
------------------------------------------------------------------------------
RESULTADO: 8/8 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion02.md` a `auditorias/auditoria_sesion02.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué? R.- Lo que más me costó fue entender la diferencia entre los códigos HTTP 404 y 405. Aprendí que el 404 indica que el libro no existe, mientras que el 405 significa que la ruta existe, pero no permite utilizar ese método HTTP.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?R.- Decidí utilizar un diccionario de Python para guardar los libros en memoria y una variable para asignar los identificadores automáticamente. De esta manera, pude implementar las operaciones sin utilizar una base de datos.
   - ¿Qué error detectó la verificación que tú no habías previsto?R.- La verificación me ayudó a comprobar que no era suficiente con que las rutas funcionaran. También tenía que respetar los códigos HTTP solicitados y asegurarme de que PATCH devolviera 405 en lugar de 404.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
